# CELL 0: ENVIRONMENT SETUP

In [1]:
import torch
import os

print("Setting up PyTorch Geometric for Colab...")

# 1. Get the current PyTorch and CUDA versions
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')

# 2. Build the URL to the pre-compiled files
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

# 3. Install PyG and its dependencies
os.system(f"pip install torch_geometric")
os.system(f"pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {wheel_url}")

print("✅ Installation Complete! You can now run Cell 1.")

Setting up PyTorch Geometric for Colab...
✅ Installation Complete! You can now run Cell 1.


# CELL 1: IMPORTS & SETUP

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold

print("✅ Cell 1: Libraries imported successfully!")

✅ Cell 1: Libraries imported successfully!


# CELL 2: DATA LOADING, PREPROCESSING & FEATURE SELECTION

In [3]:
print("Loading TCGA Dataset...")
url_tcga = 'https://raw.githubusercontent.com/shawkath73/GCN-implementation/refs/heads/main/data/tcga/BRCA%20mRNA%20SUBTYPE.csv'
df_tcga = pd.read_csv(url_tcga)

# Clean column names
df_tcga.columns = [col.split('|')[0] if '|' in col else col for col in df_tcga.columns]

# Separate features and labels
features = [col for col in df_tcga.columns if col not in ['Label', 'sample_id']]
X_raw = df_tcga[features].values
y_raw = df_tcga['Label'].values

# Encode Labels (0-4 for the 5 PAM50 subtypes)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

# FEATURE SELECTION: Remove low variance features (std < 0.1 means variance < 0.01) as per the paper
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X_raw)

print(f"✅ Cell 2 Complete!")
print(f"Total Patients: {len(df_tcga)}")
print(f"Original Genes: {X_raw.shape[1]} -> After Variance Filter: {X_selected.shape[1]}")
print(f"Target Classes Locked: {list(label_encoder.classes_)}")

Loading TCGA Dataset...
✅ Cell 2 Complete!
Total Patients: 875
Original Genes: 1000 -> After Variance Filter: 278
Target Classes Locked: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


# CELL 3: GRAPH CONSTRUCTION

In [4]:
def build_graph(X, y, k_neighbors=5):
    # Scale data independently for the specific fold to prevent data leakage
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Convert to PyTorch Tensors
    x_tensor = torch.tensor(X_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    # Build K-Nearest Neighbors Adjacency Matrix
    knn_matrix = kneighbors_graph(X_scaled, n_neighbors=k_neighbors, mode='connectivity', include_self=False)
    edge_index = torch.tensor(np.array(knn_matrix.nonzero()), dtype=torch.long)

    return Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

print("✅ Cell 3: Graph construction function ready!")

✅ Cell 3: Graph construction function ready!


# CELL 4: NEURAL NETWORK ARCHITECTURE

In [5]:
class BreastCancerGCN(torch.nn.Module):
    def __init__(self, num_features, hidden_1=64, hidden_2=32, num_classes=5):
        super(BreastCancerGCN, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_1)
        self.conv2 = GCNConv(hidden_1, hidden_2)
        self.classifier = torch.nn.Linear(hidden_2, num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Layer 1
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Layer 2
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Output Classification
        x = self.classifier(x)
        return F.log_softmax(x, dim=1)

print("✅ Cell 4: 2-Layer GCN Architecture defined!")

✅ Cell 4: 2-Layer GCN Architecture defined!


# CELL 5: 5-FOLD CROSS-VALIDATION PIPELINE

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []

print("Starting 5-Fold Cross-Validation...\n" + "="*50)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_selected, y_encoded)):
    # 1. Split data for this specific fold
    X_train, X_test = X_selected[train_idx], X_selected[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    # 2. Build isolated graphs
    train_graph = build_graph(X_train, y_train, k_neighbors=5)
    test_graph = build_graph(X_test, y_test, k_neighbors=5)

    # 3. Initialize Model and Optimizer
    model = BreastCancerGCN(num_features=train_graph.num_node_features, hidden_1=64, hidden_2=32, num_classes=5)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # 4. Training Loop
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        out = model(train_graph)
        loss = F.nll_loss(out, train_graph.y)
        loss.backward()
        optimizer.step()

    # 5. Evaluation Phase
    model.eval()
    with torch.no_grad():
        pred = model(test_graph).argmax(dim=1)
        correct = (pred == test_graph.y).sum().item()
        acc = correct / test_graph.y.size(0)
        fold_accuracies.append(acc)

        print(f"Fold {fold + 1} | Test Accuracy: {acc * 100:.2f}%")

# 6. Final Results
final_accuracy = np.mean(fold_accuracies) * 100
print("="*50)
print(f"🏆 FINAL 5-FOLD CV ACCURACY: {final_accuracy:.2f}%")
print("="*50)

Starting 5-Fold Cross-Validation...
Fold 1 | Test Accuracy: 76.00%
Fold 2 | Test Accuracy: 84.57%
Fold 3 | Test Accuracy: 74.86%
Fold 4 | Test Accuracy: 80.57%
Fold 5 | Test Accuracy: 76.57%
🏆 FINAL 5-FOLD CV ACCURACY: 78.51%
